# CommGuard — Dual-T4 Communication-Signal Study

## Monitoring GPU communication patterns for training detection

This notebook runs a controlled, single-host experiment on **Kaggle GPU T4 ×2** using the
[`waqasm86/CommGuard`](https://github.com/waqasm86/CommGuard) Python SDK.

### Research question

Within one Kaggle host containing two NVIDIA T4 GPUs, can content-agnostic system readings—especially
accessible NVML PCIe TX/RX readings—distinguish the included distributed transformer-training
workloads from included inference and non-training control workloads?

### Hypothesis

After calibration against controlled NCCL AllReduce payloads, distributed training will exhibit
communication-correlated temporal patterns that differ from the included inference and control
families strongly enough to generalize across **complete held-out runs**.

### Falsification criteria

The hypothesis is not supported in this session if any of the following occurs:

1. Exactly two T4 GPUs and two distinct rank-to-device bindings cannot be demonstrated.
2. NCCL collectives fail or silently fall back to CPU/Gloo/one GPU.
3. PCIe telemetry is unsupported or does not respond monotonically enough to controlled payload size.
4. Training, inference, and hard-negative controls substantially overlap.
5. grouped performance collapses compared with random-window performance;
6. held-out adversarial training families produce high false-negative rates;
7. results are unstable across separate Kaggle sessions.

### Scope and claims

This is an independent, unofficial prototype. It is not affiliated with or endorsed by SPAR, ERA,
UChicago XLab, William Fowler, or the cited authors and institutions.

Kaggle's two T4 GPUs are a **small-scale PCIe testbed**. This notebook does not validate multi-node,
8-GPU-node, NVLink, NVSwitch, RoCE, InfiniBand, frontier-scale, privacy, production-security, or
treaty-verification claims. NVML PCIe values are treated as **PCIe traffic readings**, not exact NCCL
byte counts or a complete measurement of inter-GPU communication.

## Experimental design

The notebook follows an evidence-first sequence:

1. Pin and install a reviewed CommGuard source revision.
2. Save software, hardware, topology, and package provenance.
3. Run CPU-safe repository tests.
4. Apply a strict dual-T4 preflight gate.
5. Prove two-rank NCCL participation using distinct GPU UUIDs.
6. Calibrate PCIe readings against known AllReduce payload sizes.
7. Run a benign and hard-negative workload corpus.
8. Run bounded adversarial training variants already implemented in CommGuard.
9. Derive deterministic windows.
10. Evaluate with whole-run grouping and signal-family ablations.
11. Export raw evidence, derived features, evaluation results, and a report.

The primary unit of independence is a **complete run**, not a telemetry row or adjacent window.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/waqasm86/CommGuard.git"

# Replace "main" with a reviewed commit SHA before producing research results.
GIT_REF = "main"

REPO = Path("/kaggle/working/CommGuard")
ARTIFACTS = Path("/kaggle/working/commguard-artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
print("Notebook run:", RUN_ID)
print("Repository:", REPO)
print("Artifacts:", ARTIFACTS)

## 1. Acquire and install the reviewed source

Internet must be enabled for this cell. For an offline Kaggle notebook, upload a clean repository
snapshot as a Kaggle Dataset and set `REPO` to that read-only dataset path, or copy it into
`/kaggle/working` first.

The install uses `--no-deps` so the notebook does not replace Kaggle's CUDA/PyTorch stack.

In [ ]:
import importlib
import importlib.util
import site

if REPO.exists() and not (REPO / ".git").exists():
    # A stale non-Git directory can be left by an interrupted Kaggle run.
    shutil.rmtree(REPO)

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO)],
        check=True,
    )

subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)

COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-build-isolation",
        "--no-deps",
        "-e",
        str(REPO),
    ],
    check=True,
)

# Kaggle/Jupyter does not always process a newly-created editable-install .pth
# file inside the already-running kernel. Add the src tree explicitly.
SRC = (REPO / "src").resolve()
if not SRC.is_dir():
    raise RuntimeError(f"Expected source directory is missing: {SRC}")

site.addsitedir(str(SRC))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

importlib.invalidate_caches()

spec = importlib.util.find_spec("commguard")
if spec is None:
    raise ModuleNotFoundError(
        "CommGuard was installed but is still not importable. "
        f"Expected module under {SRC}; current sys.path begins with {sys.path[:5]}"
    )

import commguard

print("Pinned commit:", COMMIT)
print("CommGuard import:", spec.origin)
print("CommGuard version:", commguard.__version__)

### Why the explicit `src` path is necessary

Kaggle keeps the notebook kernel alive while `pip` runs in a subprocess. A newly-created editable-install `.pth` file may therefore not be processed until the kernel restarts. This notebook explicitly adds `CommGuard/src` to `sys.path`, invalidates import caches, and verifies that the imported module comes from the pinned checkout.


## 2. Record basic system provenance

This cell records raw command output separately from CommGuard's structured preflight artifact.

In [ ]:
def run_capture(args):
    completed = subprocess.run(args, text=True, capture_output=True)
    return {
        "command": args,
        "return_code": completed.returncode,
        "stdout": completed.stdout,
        "stderr": completed.stderr,
    }

provenance = {
    "notebook_run_id": RUN_ID,
    "repository_url": REPO_URL,
    "git_ref_requested": GIT_REF,
    "commit": COMMIT,
    "python": sys.version,
    "nvidia_smi_list": run_capture(["nvidia-smi", "-L"]),
    "nvidia_smi_topology": run_capture(["nvidia-smi", "topo", "-m"]),
    "nvidia_smi_query": run_capture([
        "nvidia-smi",
        "--query-gpu=index,name,uuid,pci.bus_id,driver_version,memory.total",
        "--format=csv,noheader",
    ]),
}

(ARTIFACTS / "notebook-provenance.json").write_text(
    json.dumps(provenance, indent=2), encoding="utf-8"
)

print(provenance["nvidia_smi_list"]["stdout"])
print(provenance["nvidia_smi_topology"]["stdout"])

In [ ]:
import importlib
import torch
import commguard

commguard = importlib.reload(commguard)
module_path = Path(commguard.__file__).resolve()

if SRC not in module_path.parents:
    raise RuntimeError(
        f"Imported CommGuard from an unexpected location: {module_path}; "
        f"expected it under {SRC}"
    )

print("CommGuard:", commguard.__version__)
print("CommGuard source:", module_path)
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("NCCL version:", torch.cuda.nccl.version() if torch.cuda.is_available() else None)

for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(
        index,
        props.name,
        f"CC {props.major}.{props.minor}",
        f"{props.total_memory / 2**30:.2f} GiB",
    )

## 3. Run CPU-safe acceptance tests

These tests verify package imports, artifact schemas, create-only storage, telemetry failure semantics,
feature extraction, calibration analysis, grouped splitting, and CPU handling of distributed code.
They do not prove that the Kaggle GPUs or NCCL path work.

In [ ]:
test_env = os.environ.copy()
test_env["PYTHONPATH"] = str(SRC) + os.pathsep + test_env.get("PYTHONPATH", "")

test_command = [
    sys.executable,
    "-m",
    "pytest",
    "-m",
    "not gpu and not multigpu and not network and not slow",
    "-q",
]
subprocess.run(test_command, cwd=REPO, env=test_env, check=True)

## 4. Strict dual-T4 environment gate

The experiment stops unless CommGuard observes exactly two NVIDIA T4 GPUs, CUDA, NCCL, topology
information, and required telemetry capabilities. Installation alone is not evidence of readiness.

In [ ]:
from commguard.environment.preflight import check_environment, summarize_environment

environment = check_environment(
    strict=True,
    output=ARTIFACTS,
    check_network=False,
)
print(summarize_environment(environment))
print("\nTopology:\n", environment["topology"]["stdout"])

assert len(environment["gpus"]) == 2
assert all("T4" in gpu["name"] for gpu in environment["gpus"])

## 5. Two-rank NCCL participation proof

This is the minimum validity check for all results labelled dual-GPU:

- `WORLD_SIZE == 2`
- backend is NCCL
- rank 0 binds GPU 0
- rank 1 binds GPU 1
- GPU UUIDs are distinct
- both ranks finish a CUDA AllReduce
- rank evidence and the participation summary are preserved

In [ ]:
from commguard.orchestrator import run_experiment

smoke = run_experiment(
    "collective_all_reduce_1mib",
    output=ARTIFACTS,
    overrides={
        "iterations": 4,
        "burst_iterations": 4,
        "iteration_interval_s": 0.2,
    },
    timeout_s=180,
)

print("Smoke run:", smoke["run_id"])
print("Exit:", smoke["manifest"]["exit_status"])
print("Participation valid:", smoke["manifest"]["participation_valid"])

assert smoke["manifest"]["exit_status"] == "completed"
assert smoke["manifest"]["participation_valid"] is True

## 6. Communication-signal calibration

Calibration is a prerequisite, not a cosmetic benchmark. Controlled AllReduce payloads are swept
across several sizes and repeated. CommGuard keeps three evidence channels separate:

1. nominal tensor payload;
2. PyTorch/NCCL execution evidence;
3. NVML PCIe readings.

A negative calibration result is a valid finding and blocks detector claims.

In [ ]:
from commguard.orchestrator import run_calibration_sweep

CALIBRATION_PAYLOADS_MIB = (1, 4, 16, 64, 256)
CALIBRATION_REPETITIONS = 3

calibration = run_calibration_sweep(
    output=ARTIFACTS,
    payload_mib=CALIBRATION_PAYLOADS_MIB,
    collective="all_reduce",
    repetitions=CALIBRATION_REPETITIONS,
    timeout_s=240,
)

print("Calibration status:", calibration["status"])
print("Falsification reasons:", calibration["falsification_reasons"])
print("Rank correlation:", calibration.get("rank_correlation"))
print("Dynamic range:", calibration.get("dynamic_range"))
print("Result:", calibration["path"])

CALIBRATION_SUPPORTED = calibration["status"] == "supported"

### Calibration decision

Do not continue to classifier collection when calibration is unsupported. In that case, export the
negative evidence and report that the accessible signal did not pass the session-specific gate.

In [ ]:
if not CALIBRATION_SUPPORTED:
    raise RuntimeError(
        "Calibration was not supported in this Kaggle session. "
        "Stop detector collection, inspect the saved evidence, and report the negative result."
    )

## 7. Inspect the bounded workload corpus

### Benign training

- ordinary full-parameter DDP training

### Benign inference

- single-GPU prefill
- independent two-GPU prefill
- independent decode
- synchronized inference

### Hard-negative controls

- compute-only matrix work
- host/device transfer
- model loading

### Bounded adversarial training variants

- gradient accumulation / less frequent synchronization
- idle padding
- parameter-efficient training

These variants test whether reduced or temporally disguised communication makes training harder to
detect. CommGuard currently does **not** implement full DiLoCo. DiLoCo-style local training with a
large number of inner steps should be treated as a later extension, not claimed from this notebook.

In [ ]:
from commguard.workloads import list_workloads
from pprint import pprint

workloads = list_workloads()
for name, cfg in workloads.items():
    print(
        f"{name:38s}",
        f"label={cfg['label']:11s}",
        f"family={cfg['family']:30s}",
        f"designation={cfg['designation']}",
    )

## 8. Estimate experiment cost before using GPU time

The standard profile contains the benign corpus and the three implemented adversarial training
families. Three repetitions are the minimum meaningful starting point for whole-run evaluation.

In [ ]:
from commguard.orchestrator import estimate_matrix

STANDARD_REPETITIONS = 3
estimate = estimate_matrix("standard", repetitions=STANDARD_REPETITIONS)
print(json.dumps(estimate, indent=2))

## 9. Run the standard corpus

This is deliberately opt-in. Review the estimate above first.

The matrix randomizes bounded run order. OOMs, timeouts, telemetry failures, and rank failures are
preserved as evidence; the SDK does not silently shrink batch size or relabel a failed run.

In [ ]:
from commguard.orchestrator import run_matrix

RUN_STANDARD_CORPUS = False  # Change to True only after reviewing all previous cells.

matrix_summary = None
if RUN_STANDARD_CORPUS:
    matrix_summary = run_matrix(
        profile="standard",
        output=ARTIFACTS,
        repetitions=STANDARD_REPETITIONS,
        negative_calibration_mode=False,
        timeout_s=300,
    )
    print(json.dumps(matrix_summary, indent=2))
else:
    print("Standard corpus skipped. Set RUN_STANDARD_CORPUS=True and rerun this cell.")

## 10. Optional second-stage red-team corpus

The standard profile already includes gradient accumulation, idle padding, and parameter-efficient
training. A later CommGuard version should add:

- segmented training;
- alternating training/inference phases;
- staggered rank computation;
- sparse or delayed synchronization;
- inference-like synthetic tensor exchange;
- a Kaggle-scaled local-SGD/DiLoCo analogue.

The DiLoCo paper uses many local inner steps before outer communication; its default reported setting
uses `H = 500`, and it reports communication reductions up to 500× in its larger study. Kaggle's
two-GPU single-host experiment cannot reproduce that infrastructure or establish the same result.

## 11. Derive deterministic feature windows

Feature derivation uses complete run manifests and excludes startup by default. Windows of 5, 15, and
30 seconds test sensitivity to temporal scale.

No evaluation should proceed when there are too few successful, independent runs.

In [ ]:
from commguard.features import extract_features

feature_rows = extract_features(
    ARTIFACTS,
    output=ARTIFACTS,
    window_lengths=(5, 15, 30),
)
print("Feature rows:", len(feature_rows))

successful_run_ids = sorted({
    row["run_id"]
    for row in feature_rows
    if row.get("run_id")
})
print("Runs represented:", len(successful_run_ids))

## 12. Leakage-resistant grouped evaluation

The target is **training versus included non-training workloads**, where inference and controls are
negative examples.

Evaluation must keep every window from a complete `run_id` in one split. The SDK compares:

- majority baseline;
- simple PCIe rule;
- logistic regression;
- random forest;
- PCIe-only signals;
- non-PCIe telemetry;
- combined signals;
- held-out adversarial workload families;
- abstention/uncertainty behavior.

In [ ]:
from commguard.evaluation import evaluate_detector

evaluation = None
if RUN_STANDARD_CORPUS and feature_rows:
    evaluation = evaluate_detector(
        ARTIFACTS,
        output=ARTIFACTS,
    )
    print(json.dumps(evaluation["ablations"], indent=2)[:8000])
else:
    print("Evaluation skipped until the standard corpus has been collected.")

## 13. Evidence-grounded report

The generated report should clearly separate:

- measured observations;
- detector results;
- falsification findings;
- bounded adversarial results;
- limitations;
- untested hypotheses;
- recommendations for larger server-grade experiments.

In [ ]:
from commguard.reporting import generate_report

report_path = ARTIFACTS / "report.md"
report_text = generate_report(ARTIFACTS, output=report_path)

print("Report:", report_path)
print(report_text[:5000])

## 14. Session comparison requirement

A single Kaggle session is insufficient for generalization claims. Repeat this notebook in at least
three separate sessions while pinning the same source commit and experiment configuration.

Recommended use:

- Session A: development and threshold selection
- Session B: validation
- Session C: final untouched test

Do not pool windows randomly across sessions. Save each session's fingerprint and evaluate the final
session as a complete holdout.

In [ ]:
session_summary = {
    "notebook_run_id": RUN_ID,
    "commit": COMMIT,
    "session_fingerprint": environment["session_fingerprint"],
    "calibration_status": calibration["status"],
    "standard_corpus_executed": RUN_STANDARD_CORPUS,
    "matrix_summary": matrix_summary,
    "feature_rows": len(feature_rows),
    "evaluation_created": evaluation is not None,
}
(ARTIFACTS / "notebook-session-summary.json").write_text(
    json.dumps(session_summary, indent=2),
    encoding="utf-8",
)
print(json.dumps(session_summary, indent=2))

## 15. Export complete evidence

The archive should contain raw telemetry, manifests, rank events, logs, calibration results, derived
features, split assignments, evaluation results, provenance, and the report.

Do not publish confidential data or claim that the archive proves treaty-grade verification.

In [ ]:
from commguard.artifacts import ArtifactStore

archive_path = Path("/kaggle/working") / f"commguard-{RUN_ID}-{COMMIT[:12]}.tar.gz"
archive = ArtifactStore(ARTIFACTS).export(archive_path)
print("Exported:", archive)
print("Size MiB:", Path(archive).stat().st_size / 2**20)

## Interpretation template

Complete this only after collecting real evidence.

### Calibration

- Were PCIe readings supported on both T4 GPUs?
- Did the signal respond monotonically to payload size?
- What were the uncertainty and run-to-run variance?
- Did any payload group behave anomalously?

### Benign discrimination

- Which signal family performed best on complete held-out runs?
- How much did results deteriorate relative to random-window splitting?
- Which inference or control family caused the most false positives?
- Was detector performance stable across window sizes?

### Adversarial evaluation

- Which bounded training variant caused the most false negatives?
- What throughput, utilization, or wall-clock penalty did that variant impose?
- Did abstention capture ambiguous runs?
- Did any hard negative resemble training more strongly than inference did?

### Conclusion

State only what the Kaggle dual-T4 evidence supports. Frame successful results as a small-scale
feasibility finding and failed results as information about the limitations of accessible telemetry.

## Connection to the broader verification architecture

This notebook evaluates a narrow communication-correlated signal. It does not implement the broader
low-trust architecture described by Naci Cankaya, which separates evidence capture from evidence
evaluation, proposes cryptographic commitments, random retrospective challenges, controlled
disclosure, redundant trusted computing bases, physical monitoring, and possible network-tap or
memory-challenge mechanisms.

CommGuard is best viewed as an **inspection-method experiment** that could eventually operate inside
a larger evidence-evaluation system—not as the full evidence-capture or treaty-verification system.